# IPC2BNS-Verify — Phase 4: Hard-Constraint Verifier Layer & Stage 3 Ablation

This notebook demonstrates the core innovation of the research paper:
1. **Layer 1: Hard Citation-Existence Gating** (`citation_check.py`) — Rejects phantom sections.
2. **Layer 2: Semantic Entity Grounding** (`entity_grounding.py`) — Flags ungrounded penal claims.
3. **Repeal Veto Engine**: Detects and overrides citations of repealed sections (§124A sedition, §497 adultery, §377).
4. **Stage 3 Ablation (+Verifier)**: Benchmark run saving to `results/stage3/stage3_verifier_results.json`.
5. **Stress-Test Evaluation**: Computes Hallucination Catch Rate and False Positive Rate on adversarial dataset.
6. **Full Automated Test Suite**: Executes all 63 unit tests.

---
## 1. Mount Google Drive & Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
PROJECT_ROOT = '/content/drive/MyDrive/NLP_rspaper'
os.environ['IPC2BNS_PROJECT_ROOT'] = PROJECT_ROOT

if os.path.join(PROJECT_ROOT, 'code') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'code'))

print('Project Root:', PROJECT_ROOT)
print('Environment initialized.')

---
## 2. Install Pytest

In [ ]:
!pip install -q pytest
print('Pytest ready.')

---
## 3. Layer 1 Verification Demo: Closed-Set Citation Gating

In [ ]:
from src.verifier.citation_check import get_citation_verifier

verifier = get_citation_verifier()
test_citations = [
    [{'act': 'BNS', 'section': '103', 'raw': '[BNS §103]'}],       # Valid
    [{'act': 'BNS', 'section': '999', 'raw': '[BNS §999]'}],       # Hallucinated
    [{'act': 'IPC', 'section': '124A', 'raw': '[IPC §124A]'}],     # Repealed Sedition
    [{'act': 'IPC', 'section': '497', 'raw': '[IPC §497]'}],       # Repealed Adultery
]

for c in test_citations:
    res = verifier.verify_citations(c)
    print(f'Citation: {c[0]["raw"]:12s} -> Valid: {res.is_valid}')
    if res.rejection_reasons:
        print(f'  Reasons: {res.rejection_reasons}')

---
## 4. Master Verifier & Repeal Veto Showcase

In [ ]:
from src.verifier.verifier_pipeline import verify_answer
from src.generation.prompt_template import LegalPromptBuilder

cases = [
    ('Valid Murder Answer', 'Under [BNS §103], whoever commits murder shall be punished with death or imprisonment for life and fine.'),
    ('Hallucinated Section', 'Extortion is strictly governed under [BNS §999] with up to 10 years imprisonment.'),
    ('Repealed Sedition Claim', 'Sedition remains an offence under [IPC §124A] for inciting disaffection against the Government.'),
    ('Ungrounded Penalty', 'Under [BNS §303], simple theft carries mandatory death penalty without parole.')
]

mock_chunks = [{
    'act': 'BNS', 'section_number': '103',
    'section_title': 'Punishment for murder',
    'section_text': 'Whoever commits murder shall be punished with death or imprisonment for life and fine.'
}]

for label, text in cases:
    print('='*75)
    print(f'CASE: {label}')
    print('='*75)
    cits = LegalPromptBuilder.extract_citations(text)
    v_res = verify_answer(text, cits, mock_chunks)
    print(f'Verdict       : {v_res.verdict}')
    print(f'Is Verified   : {v_res.is_verified}')
    print(f'Final Output  :\n{v_res.verified_output_text}')
    if v_res.warnings:
        print(f'Warnings      : {v_res.warnings}')
    print()

---
## 5. Execute Stage 3 (+Verifier) Benchmark Run

In [ ]:
from src.generation.run_stage3 import run_stage3_benchmark, evaluate_verifier_stress_test

benchmark_dev = os.path.join(PROJECT_ROOT, 'data/03_benchmark/benchmark_dev.csv')
injected_errors = os.path.join(PROJECT_ROOT, 'data/03_benchmark/injected_errors.csv')
stage3_out = os.path.join(PROJECT_ROOT, 'results/stage3/stage3_verifier_results.json')

stage3_data = run_stage3_benchmark(benchmark_dev, stage3_out)
stress_metrics = evaluate_verifier_stress_test(injected_errors)

print('\n' + '='*60)
print('STAGE 3 VERIFIER STRESS-TEST METRICS')
print('='*60)
print(f'Hallucination Catch Rate : {stress_metrics["hallucination_catch_rate"]*100:.1f}%')
print(f'False Positive Rate (FPR): {stress_metrics["false_positive_rate"]*100:.1f}%')

---
## 6. Run Full Test Suite (63 Unit Tests)

In [ ]:
test_dir = os.path.join(PROJECT_ROOT, 'code/tests')
!python -m pytest "{test_dir}" -v --color=yes

---
## 7. Check WBS Progress

In [ ]:
!python "{PROJECT_ROOT}/check_progress.py" --root "{PROJECT_ROOT}" --write-report